# Privacy Filter BR v2 — Fine-tuning

Base: `openai/privacy-filter` (1.5B / 50M active MoE)

Differences vs v1:
- 18 categories (6 e-commerce additions: ORDER_ID, TRACKING_CODE, INVOICE_NUMBER, CLIENT_REVENUE, TRANSACTION_ID, CUSTOMER_ID)
- 73 output labels (was 53)
- Dataset: ~52k synthetic (was 11k in v1)

**Recommended runtime: A100 (40GB).** With 52k examples × 3 epochs, expect ~45-60min.

## Steps
1. Upload `dataset_br_v2.jsonl` and `dataset_br_v2_holdout.jsonl` to `/content/`
2. Upload `finetune_v2.py` to `/content/`
3. Run all cells

## 1. Install deps

In [ ]:
!pip install -q transformers peft datasets seqeval accelerate

## 2. Verify GPU

In [ ]:
import torch
print('CUDA:', torch.cuda.is_available())
print('Device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')
print('bf16 supported:', torch.cuda.is_bf16_supported() if torch.cuda.is_available() else False)

## 3. Verify dataset files are uploaded

In [ ]:
!ls -la /content/dataset_br_v2*.jsonl /content/finetune_v2.py
!wc -l /content/dataset_br_v2*.jsonl

## 4. Run fine-tuning

Conservative settings: 3 epochs, batch=16, LR=2e-4.

If F1 stalls, try `--epochs 5` or `--batch-size 8` to reduce memory pressure.

In [ ]:
!cd /content && python finetune_v2.py \
    --train-file /content/dataset_br_v2.jsonl \
    --eval-file /content/dataset_br_v2_holdout.jsonl \
    --output-dir /content/privacy-filter-br-v2 \
    --epochs 3 \
    --batch-size 16 \
    --lr 2e-4

## 5. Inspect benchmark report

In [ ]:
!cat /content/privacy-filter-br-v2/benchmark.txt

## 6. Sanity check: inference on a sample

In [ ]:
from transformers import AutoTokenizer, AutoModelForTokenClassification, pipeline
import torch

MODEL_DIR = '/content/privacy-filter-br-v2'
tok = AutoTokenizer.from_pretrained(MODEL_DIR)
model = AutoModelForTokenClassification.from_pretrained(MODEL_DIR, dtype=torch.float32).cuda().eval()

ner = pipeline('token-classification', model=model, tokenizer=tok, aggregation_strategy='simple', device=0)

samples = [
    'João Silva, CPF 123.456.789-00, comprou no pedido ML-2024-789456 com rastreio BR123456789BR.',
    'Empresa Acme Ltda, CNPJ 12.345.678/0001-90, faturou R$ 50.000,00 em novembro.',
    'Fatura FAT-2025-456 paga via PIX (transação E612384341436987477) por cliente CUST-998877.',
]
for s in samples:
    print('TEXT:', s)
    for ent in ner(s):
        print(f'  {ent["entity_group"]:25} | {ent["word"]!r} | score={ent["score"]:.3f}')
    print()

## 7. Package for download

Compress the merged model (~2.5GB) and download.

In [ ]:
!cd /content && zip -r privacy-filter-br-v2.zip privacy-filter-br-v2/
!ls -lh /content/privacy-filter-br-v2.zip

from google.colab import files
files.download('/content/privacy-filter-br-v2.zip')